# Atmospheric Rivers Pipeline — End-to-End Demo

Reviewer-facing walkthrough of the two-stage pipeline used in the manuscript on atmospheric rivers and cascading multi-hazards under SSP2-4.5 / SSP3-7.0 / SSP4-6.0 / SSP5-8.5.

**Scope.** This notebook runs the full code path on a single-month subset of ERA5 (January 1998) so it executes in roughly 5-10 minutes. The numerical results are *not* scientifically meaningful at this sample size — the production runs use 1980-2014 for training and 2015-2024 for holdout (see `README.md`). The intent is to demonstrate the pipeline's machinery: lazy streaming of ARCO-ERA5, Holton-dynamics features, Guan-Waliser detection, Stage 1 LightGBM emulation, event extraction, per-event topography aggregation, and Stage 2 quantile-regression hazard prediction.

Run with `jupyter nbconvert --execute notebooks/e2e_demo.ipynb`.


## 0. Setup


In [ ]:
import logging

import matplotlib.pyplot as plt
import pandas as pd

from src import config
from src.data import event_dataset
from src.features import ar_detection, physics_pipeline, topography
from src.figures import _style
from src.models import event_post

_style.apply_nature_style()
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")
print("config bbox:", config.BBOX)
print("pressure levels:", config.PRESSURE_LEVELS)


## 1. Open ARCO-ERA5 lazily, slice to one month

Single-month subset chosen so the demo completes in minutes; the lazy graph itself is identical to the production multi-decade ingest.


In [ ]:
ds = physics_pipeline.open_arco_era5()[list(physics_pipeline.required_era5_vars())]
ds_month = ds.sel(time=slice("1998-01-01", "1998-01-31"))
ds_month


## 2. Holton dynamics features


In [ ]:
feats = physics_pipeline.calculate_dynamics(ds_month)
feats


## 3. Guan-Waliser AR detection

Climatology is computed over the one-month window for demo speed; production uses 1980-2014.


In [ ]:
clim = ar_detection.compute_ivt_climatology(
    feats["ivt"], period=("1998-01-01", "1998-01-31")
)
mask = ar_detection.compute_ar_mask(
    feats["ivt"], feats["ivt_u"], feats["ivt_v"], clim
)
mask


## 4. Materialise an AR-conditional IVT field

Without trained Stage 1 boosters cached locally, we use the GW-masked IVT field as a stand-in. Swap in the production model by uncommenting the `predict_intensity` call below once `data/cache/stage1_models/` is populated.


In [ ]:
intensity = (mask.astype("float32") * feats["ivt"]).rename("ar_intensity").compute()

# Production alternative (uncomment once boosters are cached):
# from src.inference.cmip6_apply import load_stage1_boosters, predict_intensity
# boosters = load_stage1_boosters()
# intensity = predict_intensity(boosters, feats).compute()

intensity.isel(time=20).plot(figsize=(4.5, 3.0))
plt.title("AR-conditional IVT (kg m$^{-1}$ s$^{-1}$) at one timestep")
plt.show()


## 5. Extract discrete AR events


In [ ]:
topo = topography.open_era5_topography().compute()
events = event_post.extract_events(
    intensity, land_sea_mask=topo["land_sea_mask"], threshold=config.STAGE1_EVENT_THRESHOLD
)
print(f"detected {len(events)} events in this month")
events.head(10)


## 6. Per-event topography aggregates


In [ ]:
topo_feats = topography.event_topo_features(events, topo, intensity)
topo_feats.head(10)


## 7. Stage 2 dataset assembly (ERA5 precip integration)


In [ ]:
import xarray as xr

era5_full = xr.open_zarr(
    config.ERA5_ZARR_URL, consolidated=True, storage_options={"token": "anon"}, chunks={}
)
tp = physics_pipeline._apply_regional_bbox(
    era5_full[config.ERA5_SURFACE_VARS["tp"]].to_dataset()
)[config.ERA5_SURFACE_VARS["tp"]]

dataset = event_dataset.build_dataset(events, intensity, topo, tp)
dataset.head(10)


## 8. Summary

If you got here without an error, the pipeline executed end-to-end on January 1998. The numbers are illustrative only at this sample size.

Production reproduction:

```bash
python -m src.data.era5_train_assembly --period 1980-2014
python -m src.models.stage1_ar_emulator --cv
python -m src.data.event_dataset --intensity ...
python -m src.models.stage2_cascade --cv
python -m src.inference.cmip6_apply --source MPI-ESM1-2-HR --experiment ssp370 --mode direct
python -m src.inference.cmip6_apply --source MPI-ESM1-2-HR --experiment ssp370 --mode delta
python -m src.figures.fig3_ssp_shift
```
